## 1. Setup
Paths and the league-code mapping between our two sources:
football-data.co.uk uses `E0/SP1/D1/I1/F1`, the football-data.org API
uses `PL/PD/BL1/SA/FL1`. We standardise on the API codes everywhere.

In [1]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

# Notebook lives in notebooks/, data lives one level up
RAW_DIR = Path("../data/raw")

# CSV league code -> API competition code (the shared vocabulary of both sources)
LEAGUE_MAP = {
    "E0": "PL",
    "SP1": "PD",
    "D1": "BL1",
    "I1": "SA",
    "F1": "FL1",
}
CSV_SEASONS = ["2425", "2526"]

## 2. Load all odds CSVs into one frame
The ten files do **not** share a schema (119–132 columns; bookmakers
appear and disappear between seasons). `pd.concat` takes the union of
columns, so season-exclusive columns become NaN elsewhere — we measure
that in the next cell instead of letting it surprise us later.
Provenance (`league`, `season_file`) is attached via concat keys.

In [5]:
# Read each file under a (league, season) key; concat turns the keys
# into index levels, which we then promote to ordinary columns.

frames = {}
for season in CSV_SEASONS:
    for csv_code, api_code in LEAGUE_MAP.items():
        filepath = RAW_DIR / f"odds_{csv_code}_{season}.csv"
        frames[(api_code, season)] = pd.read_csv(filepath)

# .copy() defragments the frame after the union-concat,
# avoiding pandas' PerformanceWarning on subsequent inserts
odds_raw = pd.concat(frames, names=["league", "season_file"], sort=False).copy()
odds_raw = odds_raw.reset_index(level=["league", "season_file"]).reset_index(drop=True)
print(odds_raw.shape)

(3504, 152)


## 3. Missingness audit
Ranking columns by share of missing values separates two phenomena:
columns that exist only in some files (season/league schema drift,
~35–65% missing) versus genuinely sparse values. Anything we keep for
analysis must be near-complete across *all* leagues and seasons.
Findings: `Referee` (E0-only, 78%), William Hill (~62%), Coral,
Ladbrokes, 1xBet, BetVictor all too patchy to use.

In [9]:
# Share of NaN per column, worst first — the shape of the schema drift
null_share = odds_raw.isna().mean().sort_values(ascending=False)
print(null_share.head(25).round(3))

Referee    0.783
CLCD       0.650
CLCA       0.650
CLCH       0.650
CLD        0.635
CLH        0.635
CLA        0.635
LBD        0.626
LBH        0.626
LBA        0.626
LBCD       0.626
LBCH       0.626
LBCA       0.626
WHCA       0.622
WHCH       0.622
WHCD       0.622
WHD        0.622
WHA        0.622
WHH        0.622
1XBA       0.513
1XBD       0.513
1XBH       0.513
BVCH       0.511
BVCA       0.511
BVCD       0.511
dtype: float64


In [7]:
# Targeted completeness check on keep-list candidates.
# Rule: a bookmaker only stays in the analysis if its closing 1X2 odds
# are near-complete across ALL leagues and both seasons — otherwise
# cross-league margin comparisons would silently compare different samples.
keep_candidates = [
    "B365CH", "B365CD", "B365CA",   # Bet365 closing 1X2
    "PSCH", "PSCD", "PSCA",         # Pinnacle closing 1X2
    "BWCH", "BWCD", "BWCA",         # Bwin closing 1X2
    "MaxCH", "MaxCD", "MaxCA",      # market maximum, closing
    "AvgCH", "AvgCD", "AvgCA",      # market average, closing
]
print(odds_raw[keep_candidates].isna().mean().round(4))

B365CH    0.0000
B365CD    0.0000
B365CA    0.0000
PSCH      0.2437
PSCD      0.2437
PSCA      0.2437
BWCH      0.1909
BWCD      0.1909
BWCA      0.1909
MaxCH     0.0000
MaxCD     0.0000
MaxCA     0.0000
AvgCH     0.0000
AvgCD     0.0000
AvgCA     0.0000
dtype: float64


In [10]:
# Where exactly are Pinnacle and Bwin missing? Random scatter is tolerable;
# concentration by league/season means the column can't support cross-league work.
missing_structure = (
    odds_raw
    .groupby(["league", "season_file"])[["PSCH", "BWCH"]]
    .apply(lambda g: g.isna().mean())
    .round(3)
)
print(missing_structure)

                     PSCH   BWCH
league season_file              
BL1    2425         0.000  0.382
       2526         0.513  0.000
FL1    2425         0.000  0.382
       2526         0.500  0.000
PD     2425         0.000  0.397
       2526         0.505  0.003
PL     2425         0.000  0.371
       2526         0.447  0.000
SA     2425         0.000  0.374
       2526         0.479  0.000


In [11]:
# Is the missingness early-season or late-season? This decides whether our
# calendar-2025 window is affected at all.
tmp = odds_raw.copy()
tmp["date_parsed"] = pd.to_datetime(tmp["Date"], dayfirst=True, format="mixed")
tmp["in_2025"] = tmp["date_parsed"].dt.year == 2025

coverage = (
    tmp.groupby(["season_file", "in_2025"])[["PSCH", "BWCH"]]
    .apply(lambda g: g.isna().mean())
    .round(3)
)
print(coverage)
print("\nRows in calendar 2025:", tmp["in_2025"].sum())

                      PSCH   BWCH
season_file in_2025              
2425        False    0.000  0.000
            True     0.000  0.715
2526        False    0.898  0.001
            True     0.001  0.000

Rows in calendar 2025: 1736


## 4. Keep-list and tidy odds table
Decision trail: WH/Coral/Ladbrokes/1xBet/BetVictor dropped (coverage),
Bwin dropped (71.5% missing Jan–May 2025, inside the analysis window),
Pinnacle retained (its gap is Jan–May 2026, outside the window).
Final: Bet365 + Pinnacle closing 1X2 + market Max/Avg + match core,
filtered to calendar-2025 kickoffs.

In [12]:
KEEP_COLS = {
    # match core
    "league": "league",
    "season_file": "season_file",
    "Date": "date",
    "Time": "kickoff_local",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR": "result",
    # closing 1X2 odds
    "B365CH": "b365_home", "B365CD": "b365_draw", "B365CA": "b365_away",
    "PSCH": "pin_home",   "PSCD": "pin_draw",   "PSCA": "pin_away",
    "MaxCH": "max_home",  "MaxCD": "max_draw",  "MaxCA": "max_away",
    "AvgCH": "avg_home",  "AvgCD": "avg_draw",  "AvgCA": "avg_away",
}

odds = odds_raw[list(KEEP_COLS)].rename(columns=KEEP_COLS).copy()
odds["date"] = pd.to_datetime(odds["date"], dayfirst=True, format="mixed")
odds = odds[odds["date"].dt.year == 2025].reset_index(drop=True)

print(odds.shape)                 # expect (1736, 21)
print(odds["league"].value_counts())
odds.head()

(1736, 21)
league
PL     378
PD     370
SA     368
FL1    314
BL1    306
Name: count, dtype: int64


,league,season_file,date,kickoff_local,home_team,away_team,home_goals,away_goals,result,b365_home,...,b365_away,pin_home,pin_draw,pin_away,max_home,max_draw,max_away,avg_home,avg_draw,avg_away
0,PL,2425,2025-01-01,17:30,Brentford,Arsenal,1,3,A,5.50,...,1.55,6.00,4.50,1.56,7.16,4.90,1.57,5.87,4.46,1.54
1,PL,2425,2025-01-04,12:30,Tottenham,Newcastle,1,2,A,3.90,...,1.73,3.90,4.69,1.79,3.90,4.69,1.94,3.81,4.50,1.79
2,PL,2425,2025-01-04,15:00,Aston Villa,Leicester,2,1,H,1.33,...,8.50,1.34,5.69,8.32,1.37,6.00,9.50,1.33,5.63,8.49
3,PL,2425,2025-01-04,15:00,Bournemouth,Everton,1,0,H,1.67,...,5.00,1.70,3.97,5.23,1.72,4.00,5.25,1.69,3.86,5.10
4,PL,2425,2025-01-04,15:00,Crystal Palace,Chelsea,1,1,D,3.60,...,1.91,3.64,3.94,2.00,3.89,4.00,2.05,3.58,3.84,1.98


## 5. Load API match data
The football-data.org JSONs carry structured match records: IDs, UTC
kickoff timestamps, and canonical team names. This is the "clean spine"
side of the join — the odds CSVs will be matched onto it. We keep the
API's `utcDate` (proper timezone-aware timestamps, unlike the CSVs'
local date strings) and both team-name variants (`name`, `shortName`)
as join ammunition.

In [15]:
API_COMPETITIONS = ["PL", "PD", "BL1", "SA", "FL1"]
API_SEASONS = [2024, 2025]

records = []
for comp in API_COMPETITIONS:
    for season in API_SEASONS:
        filepath = RAW_DIR / f"{comp}_{season}_matches.json"
        with open(filepath, encoding="utf-8") as f:
            payload = json.load(f)
        for m in payload["matches"]:
            records.append({
                "match_id": m["id"],
                "league": comp,
                "season_api": season,
                "utc_date": m["utcDate"],
                "status": m["status"],
                "matchday": m.get("matchday"),
                "home_name": m["homeTeam"]["name"],
                "home_short": m["homeTeam"]["shortName"],
                "away_name": m["awayTeam"]["name"],
                "away_short": m["awayTeam"]["shortName"],
                "home_goals_api": m["score"]["fullTime"]["home"],
                "away_goals_api": m["score"]["fullTime"]["away"],
            })

api = pd.DataFrame(records)
api["utc_date"] = pd.to_datetime(api["utc_date"])
api["kickoff_date"] = api["utc_date"].dt.tz_convert("Europe/London").dt.date  # see below

print(api.shape)                      # expect (3504, 13)
print(api["status"].value_counts())   # how many FINISHED vs other

(3504, 13)
status
FINISHED    3501
AWARDED        3
Name: count, dtype: int64


In [16]:
api_2025 = api[
    (api["utc_date"].dt.year == 2025) & (api["status"] == "FINISHED")
].reset_index(drop=True)

print(api_2025.shape)
print(api_2025["league"].value_counts())

(1735, 13)
league
PL     378
PD     370
SA     368
FL1    313
BL1    306
Name: count, dtype: int64


In [17]:
# Who are the AWARDED matches, and does one of them explain the FL1 gap?
awarded = api[api["status"] == "AWARDED"]
print(awarded[["league", "utc_date", "home_name", "away_name",
               "home_goals_api", "away_goals_api"]])

     league                  utc_date           home_name         away_name  \
1640    BL1 2024-12-14 14:30:00+00:00  1. FC Union Berlin   VfL Bochum 1848   
3122    FL1 2025-03-16 16:15:00+00:00     Montpellier HSC  AS Saint-Étienne   
3496    FL1 2026-05-17 19:00:00+00:00           FC Nantes       Toulouse FC   

      home_goals_api  away_goals_api  
1640               0               2  
3122               0               2  
3496               0               0  


In [18]:
# Does the odds table have the awarded FL1 match, and with what result?
print(odds[
    (odds["league"] == "FL1")
    & (odds["home_team"].str.contains("Montpellier"))
    & (odds["date"].dt.month == 3)
][["date", "home_team", "away_team", "home_goals", "away_goals", "result",
   "b365_home", "pin_home"]])

          date    home_team   away_team  home_goals  away_goals result  \
842 2025-03-02  Montpellier      Rennes           0           4      A   
859 2025-03-16  Montpellier  St Etienne           0           2      A   

     b365_home  pin_home  
842       3.75      3.94  
859       2.30      2.30  


In [19]:
# AWARDED matches have official results and closed markets -> analytically
# equivalent to FINISHED for our purposes. Documented in the quality log.
api_2025 = api[
    (api["utc_date"].dt.year == 2025)
    & (api["status"].isin(["FINISHED", "AWARDED"]))
].reset_index(drop=True)

print(api_2025.shape)                    # expect (1736, 13) — parity with odds
print(api_2025["league"].value_counts()) # FL1 should now read 314

(1736, 13)
league
PL     378
PD     370
SA     368
FL1    314
BL1    306
Name: count, dtype: int64


## 6. Joining odds to API matches
Join key: `(league, kickoff_date, home_team, away_team)`. Team names
differ systematically between sources ("St Etienne" vs "AS Saint-Étienne"),
so the strategy is: exact join → measure failures → deterministic
name-mapping table → re-join → validate to exactly 1736 matches.
Fuzzy matching is deliberately avoided: it fails silently; a lookup
table is auditable.

In [20]:
# Minimal normalisation both sides: lowercase, strip accents, drop punctuation.
# This resolves trivial mismatches so the mapping table only has to carry
# genuine naming differences.
import unicodedata

def normalise(name: str) -> str:
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode()
    return name.lower().replace(".", "").replace("-", " ").strip()

odds["home_key"] = odds["home_team"].map(normalise)
odds["away_key"] = odds["away_team"].map(normalise)
api_2025["date_key"] = pd.to_datetime(api_2025["kickoff_date"])
api_2025["home_key"] = api_2025["home_short"].map(normalise)   # shortName ≈ CSV style
api_2025["away_key"] = api_2025["away_short"].map(normalise)

merged = odds.merge(
    api_2025,
    left_on=["league", "date", "home_key", "away_key"],
    right_on=["league", "date_key", "home_key", "away_key"],
    how="left",
    suffixes=("", "_api"),
)

matched = merged["match_id"].notna()
print(f"Matched: {matched.sum()} / {len(merged)} ({matched.mean():.1%})")

Matched: 947 / 1736 (54.6%)


In [21]:
# Which CSV names never matched? These are the mapping table's contents.
unmatched = merged.loc[~matched]
failed_names = pd.concat([
    unmatched["home_team"], unmatched["away_team"]
]).value_counts()
print(f"Unmatched rows: {len(unmatched)}")
print(failed_names.head(30))

Unmatched rows: 789
Brighton         38
Wolves           38
Nott'm Forest    38
Ath Madrid       38
Vallecano        37
Espanol          37
Sevilla          37
Sociedad         37
Betis            37
Ath Bilbao       37
Barcelona        37
Como             36
Lyon             35
Angers           35
Lens             35
Rennes           35
Ein Frankfurt    34
Bayern Munich    34
Werder Bremen    34
Paris SG         34
Venezia          20
Leicester        19
Ipswich          19
St Etienne       19
Reims            19
Valencia         18
Villarreal       18
Leeds            18
Girona           17
Celta            17
Name: count, dtype: int64


In [22]:
# Candidate pairs for the mapping table: unmatched CSV names vs API names
# in the same league. Human eyes approve every pair — no auto-accept.
csv_names = pd.concat([
    odds[["league", "home_team"]].rename(columns={"home_team": "team"}),
    odds[["league", "away_team"]].rename(columns={"away_team": "team"}),
]).drop_duplicates()

api_names = pd.concat([
    api_2025[["league", "home_short"]].rename(columns={"home_short": "team"}),
    api_2025[["league", "away_short"]].rename(columns={"away_short": "team"}),
]).drop_duplicates()

matched_keys = set(api_2025["home_key"]).union(api_2025["away_key"])
csv_names["key"] = csv_names["team"].map(normalise)
unmatched_csv = csv_names[~csv_names["key"].isin(matched_keys)]

print(f"CSV team names needing mapping: {len(unmatched_csv)}")
for _, row in unmatched_csv.iterrows():
    league_api = sorted(api_names.loc[api_names["league"] == row["league"], "team"])
    print(f"\n{row['league']}: '{row['team']}'  ->  candidates: {league_api}")

CSV team names needing mapping: 31

PL: 'Brighton'  ->  candidates: ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton Hove', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich Town', 'Leeds United', 'Leicester City', 'Liverpool', 'Man City', 'Man United', 'Newcastle', 'Nottingham', 'Southampton', 'Sunderland', 'Tottenham', 'West Ham', 'Wolverhampton']

PL: 'Wolves'  ->  candidates: ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton Hove', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich Town', 'Leeds United', 'Leicester City', 'Liverpool', 'Man City', 'Man United', 'Newcastle', 'Nottingham', 'Southampton', 'Sunderland', 'Tottenham', 'West Ham', 'Wolverhampton']

PL: 'Nott'm Forest'  ->  candidates: ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton Hove', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich Town', 'Leeds United', 'Leicester City', 'Liverpool', 'Man City', 'Man United', '

In [24]:
# Apply the deterministic name map: CSV names -> API short names,
# then rebuild join keys and re-attempt. Target: 1736/1736.
name_map = pd.read_csv("../data/team_name_map.csv", encoding="utf-8")

# dict keyed by (league, csv_name) — same name can differ per league
map_dict = {
    (r.league, r.csv_name): r.api_short_name for r in name_map.itertuples()
}

def apply_map(row_league: str, name: str) -> str:
    return map_dict.get((row_league, name), name)

odds["home_mapped"] = [apply_map(l, n) for l, n in zip(odds["league"], odds["home_team"])]
odds["away_mapped"] = [apply_map(l, n) for l, n in zip(odds["league"], odds["away_team"])]
odds["home_key"] = odds["home_mapped"].map(normalise)
odds["away_key"] = odds["away_mapped"].map(normalise)

merged = odds.merge(
    api_2025,
    left_on=["league", "date", "home_key", "away_key"],
    right_on=["league", "date_key", "home_key", "away_key"],
    how="left",
    suffixes=("", "_api"),
)

matched = merged["match_id"].notna()
print(f"Matched: {matched.sum()} / {len(merged)} ({matched.mean():.1%})")

# Whatever remains is NOT name-shaped — inspect it raw
leftovers = merged.loc[~matched, ["league", "date", "home_team", "away_team"]]
print(f"\nRemaining unmatched: {len(leftovers)}")
print(leftovers.head(20))

Matched: 1736 / 1736 (100.0%)

Remaining unmatched: 0
Empty DataFrame
Columns: [league, date, home_team, away_team]
Index: []


In [25]:
# The join matched on (league, date, teams). If any pairing is wrong,
# the two sources' scorelines will disagree. Zero disagreements = joined
# rows are verifiably the same real-world matches.
score_mismatch = merged[
    (merged["home_goals"] != merged["home_goals_api"])
    | (merged["away_goals"] != merged["away_goals_api"])
]
print(f"Score disagreements: {len(score_mismatch)}")
if len(score_mismatch):
    print(score_mismatch[["league", "date", "home_team", "away_team",
                          "home_goals", "away_goals",
                          "home_goals_api", "away_goals_api"]])

Score disagreements: 0


In [26]:
# Final tidy table: drop join scaffolding, keep analysis columns,
# write to data/processed/ as the notebook's single output artifact.
final_cols = [
    "match_id", "league", "date", "utc_date", "matchday",
    "home_mapped", "away_mapped", "home_goals", "away_goals", "result",
    "b365_home", "b365_draw", "b365_away",
    "pin_home", "pin_draw", "pin_away",
    "max_home", "max_draw", "max_away",
    "avg_home", "avg_draw", "avg_away",
]
matches = (
    merged[final_cols]
    .rename(columns={"home_mapped": "home_team", "away_mapped": "away_team"})
    .sort_values(["league", "utc_date"])
    .reset_index(drop=True)
)

out_path = Path("../data/processed/matches_2025.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
matches.to_csv(out_path, index=False)
print(f"Written: {out_path} — {matches.shape}")
matches.head()

Written: ..\data\processed\matches_2025.csv — (1736, 22)


,match_id,league,date,utc_date,matchday,home_team,away_team,home_goals,away_goals,result,...,b365_away,pin_home,pin_draw,pin_away,max_home,max_draw,max_away,avg_home,avg_draw,avg_away
0,502507,BL1,2025-01-10,2025-01-10 19:45:00+00:00,16,Dortmund,Leverkusen,2,3,A,...,1.91,3.88,3.82,1.96,3.88,3.95,2.12,3.65,3.76,1.97
1,502510,BL1,2025-01-11,2025-01-11 14:30:00+00:00,16,Freiburg,Holstein Kiel,3,2,H,...,6.50,1.46,5.04,6.73,1.47,5.18,7.59,1.44,4.87,6.60
2,502509,BL1,2025-01-11,2025-01-11 14:30:00+00:00,16,Heidenheim,Union Berlin,2,0,H,...,2.25,3.25,3.46,2.32,3.29,3.46,2.62,3.16,3.33,2.31
3,502508,BL1,2025-01-11,2025-01-11 14:30:00+00:00,16,Hoffenheim,Wolfsburg,0,1,A,...,3.00,2.36,3.48,3.15,2.45,3.60,3.15,2.33,3.44,3.02
4,502512,BL1,2025-01-11,2025-01-11 14:30:00+00:00,16,Mainz,Bochum,2,0,H,...,5.00,1.68,3.97,5.47,1.70,4.10,5.50,1.66,3.95,5.18
